# 01 Data Collection

This notebook downloads baseline index and volatility data, then stores the raw files under `data/raw/`. For actual NIFTY and BANKNIFTY futures or options research, replace or augment the Yahoo Finance proxies with broker or exchange CSV exports.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import MarketDataLoader
from src.utils import load_yaml_config, set_random_seed

config = load_yaml_config(PROJECT_ROOT / "config" / "parameters.yaml")
set_random_seed(config["project"]["random_seed"])
loader = MarketDataLoader(default_interval=config["data"]["frequency"])
raw_dir = PROJECT_ROOT / config["data"]["raw_data_dir"]
raw_dir.mkdir(parents=True, exist_ok=True)
raw_dir

In [ ]:
downloaded = {}

for symbol in config["data"]["spot_symbols"]:
    frame = loader.download_yfinance(
        ticker=symbol,
        start=config["data"]["start_date"],
        end=config["data"]["end_date"],
    )
    file_name = f"{symbol.replace('^', '').lower()}_spot.csv"
    loader.save_data(frame, raw_dir / file_name)
    downloaded[symbol] = frame.tail()

vix = loader.download_yfinance(
    ticker=config["data"]["volatility_symbol"],
    start=config["data"]["start_date"],
    end=config["data"]["end_date"],
)
loader.save_data(vix, raw_dir / "india_vix.csv")

downloaded